# Lab 3 (Fixed): Early Fusion and Classification with MLP

In [1]:

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [4]:
# Load features
image_features = np.load("coco_features/image_features.npy")
caption_features = np.load("coco_features/caption_features.npy")
labels = np.load("coco_features/labels.npy")

# Ensure both are float32 and 2D
image_features = np.array(image_features, dtype=np.float32)
caption_features = np.array(caption_features, dtype=np.float32)

print("Image features shape:", image_features.shape)
print("Caption features shape:", caption_features.shape)

# Early fusion
X = np.concatenate([image_features, caption_features], axis=1)
y = labels
print("X shape:", X.shape)
print("y shape:", y.shape)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

Image features shape: (5000, 2048)
Caption features shape: (0,)


ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s) and the array at index 1 has 1 dimension(s)

In [ ]:
# ---------------- MLP Model ----------------
class MLP(nn.Module):
    def __init__(self, input_dim=2816, hidden1=1024, hidden2=512, output_dim=80):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ---------------- Training ----------------
epochs = 10
batch_size = 64
train_losses, val_losses = [], []

In [ ]:
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for i in range(0, len(X_train), batch_size):
        xb = X_train[i:i+batch_size].to(device)
        yb = y_train[i:i+batch_size].to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / (len(X_train)//batch_size))
    
    # Validation
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(device))
        val_loss = criterion(preds, y_test.to(device)).item()
        val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_loss:.4f}")

# ---------------- Plot Loss Curves ----------------
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()